# Sampling Diverse Networks and Statements

Reproduces the diversity-maximizing selection procedure described in **Appendix C**
of the paper ("Maximin Selection of Networks and Statements"):

1. **(A) Networks**: sample a large pool of candidate graphs and greedily select
   `K=8` structurally diverse instances per network family (Erdős–Rényi, Watts–Strogatz)
   using a *maximin* criterion in a z-scored structural-metric space
   (`§Social Networks`, `Appendix C.2`).
2. **(B) Statements**: from a larger corpus of medical-indication statements
   (`savcisens2025trilemma`), select the `20` used in the experiments via the same
   maximin criterion, applied to features that summarize ground truth and
   cross-model agreement (`§Discussion Statements`, `Appendix C.1`).

**Maximin criterion**: greedily pick the point that is farthest (based on Euclidean
distance, on standardized features) from all previously selected points. This
maximizes the *minimum* pairwise distance among the selected set, favoring broad
coverage of the feature space over a random sample.


## (A) Sampling Networks for Structural Diversity

We first generate a pool of `N=1,000` candidate graphs per network family and
compute, for each candidate, a vector of structural summary statistics:

1. mean shortest-path length,
2. global clustering coefficient,
3. mean degree,
4. standard deviation of degree, and
5. degree of node `0` (the node designated as the "Clinical Physician" role slot
   in the final experiment)\*.

We then select `K=8` graphs per family using maximin in the z-scored metric space.

\* *We additionally require that node `i=0` has high variability in degree across
the selected graphs, so it does not consistently occupy the same structural
position (e.g., always a hub or always a leaf).*


In [ ]:
import os
import sys

sys.path.append(os.path.abspath('..'))

import numpy as np
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler

from src.core.metrics.network import (
    connected_components,
    global_clustering_coefficient,
    local_clustering_coefficients,
    mean_shortest_path,
)
from src.Network import Network


def maximin_select(X: np.ndarray, K: int) -> list[int]:
    """Greedily select K points from X that maximize the minimum pairwise distance."""
    selected = [int(np.argmax(np.linalg.norm(X, axis=1)))]

    for _ in range(K - 1):
        distances = cdist(X, X[selected], metric="euclidean")
        min_dist = distances.min(axis=1)
        min_dist[selected] = -np.inf  # exclude already-selected points
        next_idx = np.argmax(min_dist)
        selected.append(int(next_idx))

    return selected


def graph2vec(G: Network) -> tuple[np.ndarray, int]:
    """Summarize a Network as a structural feature vector.

    Returns:
        (mean shortest path, global clustering coefficient, mean degree,
         std degree, degree of node 0), number of connected components.
    """
    A = G.adjacency_matrix()
    mean_sp = mean_shortest_path(A)
    gcc = global_clustering_coefficient(A)
    std_deg = np.std(np.sum(A, axis=1))
    mean_deg = np.mean(np.sum(A, axis=1))
    node0_deg = np.sum(A[0, :])

    return np.array([mean_sp, gcc, mean_deg, std_deg, node0_deg]), len(connected_components(A))


# Deterministic seed pool shared by both network families.
N_AGENTS = 48  # matches n=48 in the paper (Setup, Social Networks)

master_rng = np.random.Generator(np.random.PCG64(814183))
seeds = master_rng.integers(low=0, high=2**32, size=1000, dtype=np.uint32)


### (A.1) Erdős–Rényi graphs

In [ ]:
cfg_er = {
    "seed": None,
    "network": {
        "generator": "ER",
        "params": {"n": N_AGENTS, "p": 0.3},
    },
}

X, n_components = [], []
for seed in seeds:
    cfg = cfg_er.copy()
    cfg["seed"] = int(seed)
    G = Network(cfg, remap_seed=False)
    vec, n_comp = graph2vec(G)
    X.append(vec)
    n_components.append(n_comp)
X = np.array(X)

Xz = StandardScaler().fit_transform(X)
er_selection_ids = maximin_select(Xz, K=8)

assert all(n_components[i] == 1 for i in er_selection_ids), "All selected ER graphs must be connected"

for i in er_selection_ids:
    r = X[i]
    print(
        f"{seeds[i]:>11} : mean-sp={r[0]:.3f}, glob-clustering-coeff={r[1]:.3f}, "
        f"mean-deg={r[2]:>6.3f}, std-deg={r[3]:.3f}, node0-deg={r[4]:.3f}"
    )

print("\nSelected ER seeds:", seeds[er_selection_ids].tolist())


### (A.2) Watts–Strogatz graphs

#### Step 1: calibrate `(k, β)` for small-world sampling

Watts–Strogatz networks require a `(k, β)` pair that reliably yields
small-world structure — high clustering *and* short average path length
relative to a degree-matched Erdős–Rényi baseline (small-worldness `σ = γ/λ`,
with `γ = C_WS / C_ER` and `λ = L_WS / L_ER`). We grid-search `(k, β)` and keep
the connectivity-feasible configuration with the highest `σ`.


In [ ]:
cfg_ws = {
    "seed": None,
    "network": {"generator": "WS", "params": {"n": N_AGENTS, "k": None, "beta": None}},
}

k_grid = [4, 6, 8, 10, 12]
beta_grid = [0.001, 0.003, 0.01, 0.03, 0.05, 0.1]
n_samples = 100
connectivity_threshold = 0.99

results, gammas, lambdas = [], [], []
for k in k_grid:
    for beta in beta_grid:
        cfg_ws["network"]["params"]["k"] = k
        cfg_ws["network"]["params"]["beta"] = beta

        C_ws, L_ws, C_er, L_er = [], [], [], []
        ws_connected = er_connected = 0
        for i in range(n_samples):
            cfg_ws["seed"] = int(i)
            cfg_er["seed"] = int(i + 10_000)

            A_ws = Network(cfg_ws, remap_seed=False).adjacency_matrix()
            l_ws = mean_shortest_path(A_ws)
            if l_ws is not None and np.isfinite(l_ws):
                C_ws.append(np.mean(local_clustering_coefficients(A_ws)))
                L_ws.append(l_ws)
                ws_connected += 1

            # Degree-matched ER baseline: same mean degree as the WS candidate.
            cfg_er["network"]["params"]["p"] = k / (N_AGENTS - 1)
            A_er = Network(cfg_er, remap_seed=False).adjacency_matrix()
            l_er = mean_shortest_path(A_er)
            if l_er is not None and np.isfinite(l_er):
                C_er.append(np.mean(local_clustering_coefficients(A_er)))
                L_er.append(l_er)
                er_connected += 1

        if ws_connected / n_samples < connectivity_threshold or er_connected / n_samples < connectivity_threshold:
            continue  # (k, beta) too likely to produce a disconnected graph

        gamma = float(np.mean(C_ws) / np.mean(C_er))
        lam = float(np.mean(L_ws) / np.mean(L_er))
        sigma = gamma / lam
        if np.isnan(sigma):
            continue

        gammas.append(gamma)
        lambdas.append(lam)
        results.append((k, beta, sigma, gamma, lam))

# Keep configurations with above-median clustering gain and below-median path-length
# cost, then pick the highest small-worldness among those.
gamma_min = np.quantile(gammas, 0.5)
lambda_max = np.quantile(lambdas, 0.5)
filtered = [r for r in results if r[3] >= gamma_min and r[4] <= lambda_max]
filtered.sort(key=lambda r: r[2], reverse=True)

best_k, best_beta, best_sigma, *_ = filtered[0]
print(f"Selected (k, beta) = ({best_k}, {best_beta}), small-worldness sigma = {best_sigma:.3f}")


This calibration selects `k=8, β=0.1` (the values used throughout the paper;
`§Social Networks`).

#### Step 2: sample and select 8 diverse WS graphs


In [ ]:
cfg_ws = {
    "seed": None,
    "network": {"generator": "WS", "params": {"n": N_AGENTS, "k": 8, "beta": 0.1}},
}

X, n_components = [], []
for seed in seeds:
    cfg = cfg_ws.copy()
    cfg["seed"] = int(seed)
    G = Network(cfg, remap_seed=False)
    vec, n_comp = graph2vec(G)
    X.append(vec)
    n_components.append(n_comp)
X = np.array(X)

Xz = StandardScaler().fit_transform(X)
ws_selection_ids = maximin_select(Xz, K=8)

assert all(n_components[i] == 1 for i in ws_selection_ids), "All selected WS graphs must be connected"

for i in ws_selection_ids:
    r = X[i]
    print(
        f"{seeds[i]:>11} : mean-sp={r[0]:.3f}, glob-clustering-coeff={r[1]:.3f}, "
        f"mean-deg={r[2]:>6.3f}, std-deg={r[3]:.3f}, node0-deg={r[4]:.3f}"
    )

print("\nSelected WS seeds:", seeds[ws_selection_ids].tolist())


## (B) Sampling Statements

From a larger corpus of medical-indication statements, we select the `20`
statements used in the experiments (`§Discussion Statements`). Diversity is
enforced over five features derived from zero-shot model predictions
(ground-truth label, the "medical LLM"'s predicted label and accuracy, other
models' average accuracy, and medical-vs-other agreement), with a *balanced*
maximin variant that keeps the true/false label split even.


In [ ]:
def maximin_select_balanced(
    X: np.ndarray,
    K: int,
    labels: np.ndarray,
    init_selected: list[int] | None = None,
) -> list[int]:
    """Select K points with a balanced binary label split via alternating maximin.

    Args:
        X: Feature matrix of shape (N, D).
        K: Even number of points to select.
        labels: Binary labels of shape (N,) (here: ground-truth true/false).
        init_selected: Optional pre-selected indices to seed the selection
            (used to guarantee a few canonical example statements are included).

    Returns:
        Indices of selected points, `quota` from each label class.
    """
    if K % 2:
        raise ValueError(f"K must be even, got {K}")
    quota = K // 2

    selected = list(init_selected or [])
    c0 = sum(labels[i] == 0 for i in selected)
    c1 = len(selected) - c0
    if c0 > quota or c1 > quota:
        raise ValueError("Pre-selected indices exceed per-class quota")

    if not selected:
        first = int(np.argmax(np.linalg.norm(X, axis=1)))
        selected.append(first)
        c0 += labels[first] == 0
        c1 += labels[first] == 1

    if np.sum(labels == 0) < quota or np.sum(labels == 1) < quota:
        raise ValueError("Insufficient samples to fill balanced quotas")

    while c0 < quota or c1 < quota:
        target = 0 if c0 < quota else 1
        dist = cdist(X, X[selected], metric="euclidean").min(axis=1)
        dist[selected] = -np.inf
        dist[labels != target] = -np.inf
        nxt = int(np.argmax(dist))
        selected.append(nxt)
        c0 += labels[nxt] == 0
        c1 += labels[nxt] == 1

    return selected


In [ ]:
from pathlib import Path

import polars as pl

records_path = Path("../data/resources/predictions/")

# The 15 LLMs used as simulation agents (Appendix D, Table "Summary of the 15 LLMs").
models = [
    "llama-assistant", "llama-base", "llama-biomed", "llama-chemist", "llama-coder",
    "llama-cyber", "llama-finance", "llama-hermes", "llama-law", "llama-lexicographer",
    "llama-linguist", "llama-openmath", "llama-roleplay", "llama-scholar", "llama-user",
]


In [ ]:
# Load the medical ("doctor") LLM's zero-shot predictions on the full statement corpus.
df_doc = (
    pl.read_csv(records_path / "zs_med_test_split_llama-doc.csv")
    .filter(pl.col("negation") == 0, pl.col("real_object") == 1)
    .select(
        pl.col("").alias("init_idx"),
        "statement",
        pl.col("correct").alias("label_ground_truth"),
        pl.col("predicted_label").alias("doc_predicted_label"),
        pl.col("prob_true").alias("doc_prob"),
    )
    .with_row_index("idx")
)
valid_idx = df_doc["init_idx"].to_list()

# Aggregate the other 14 models' predictions on the same statements.
dfs_models = [
    pl.read_csv(records_path / f"zs_med_test_split_{model}.csv")
    .filter(pl.col("").is_in(valid_idx))
    .rename({"": "init_idx"})
    .select("init_idx", "predicted_label", "prob_true", "statement")
    .with_columns((pl.col("predicted_label") == 1).cast(pl.Float32).alias("predicted_label"))
    for model in models
]

for i, df_model in enumerate(dfs_models):
    assert df_model["init_idx"].to_list() == valid_idx, f"Index mismatch in {models[i]}"
    statements = df_model["statement"].to_list()
    assert len(statements) == len(set(statements)), f"Duplicate statements found in {models[i]}"

df_agg = (
    pl.concat(dfs_models)
    .group_by("init_idx")
    .agg(
        pl.col("predicted_label").mean().alias("other_frac_predicted_label"),
        pl.col("prob_true").median().alias("other_median_prob"),
    )
    .with_row_index("idx")
)

df_preds = df_doc.join(df_agg, on="idx", how="inner")
assert len(df_preds) == len(df_doc), "Join mismatch between doc predictions and aggregated predictions"

# Canonical example statements (used throughout the paper's figures/text) are
# force-included via `init_selected` in the balanced maximin call below.
start_statements = [
    "Silver is indicated for the treatment of keratosis pilaris.",
    "Terbutaline is indicated for the treatment of cramps.",
    "Methyl nicotinate is indicated for the treatment of aches.",
    "Terbutaline is indicated for the treatment of asthma.",
]
start_idxs = df_preds.filter(pl.col("statement").is_in(start_statements))["idx"].to_list()
assert len(start_idxs) == len(start_statements), "Missing one or more canonical example statements"

# Diversity features: ground truth, doc model's call, doc/other accuracy, and
# doc-vs-other agreement ("consensus_score" here means agreement, not accuracy).
df_preds = df_preds.with_columns(
    (1 - (pl.col("doc_prob") - pl.col("other_median_prob")).abs()).cast(pl.Float64).alias("consensus_score"),
    (1 - (pl.col("label_ground_truth") - pl.col("other_frac_predicted_label")).abs()).cast(pl.Float64).alias("other_accuracy"),
    (1 - (pl.col("label_ground_truth") - pl.col("doc_prob")).abs()).cast(pl.Float64).alias("doc_accuracy"),
)
df_preds.head()


In [ ]:
FEATURE_COLS = ["label_ground_truth", "doc_predicted_label", "doc_accuracy", "other_accuracy", "consensus_score"]

X = df_preds.select(FEATURE_COLS).to_numpy().astype(float)
Xz = StandardScaler().fit_transform(X)

selected_ids = maximin_select_balanced(Xz, K=30, labels=X[:, 0], init_selected=start_idxs)
df_preds.filter(pl.col("idx").is_in(selected_ids)).select(["idx", "statement"] + FEATURE_COLS)


> **Note**: `K=30` is selected here as an over-sample; the paper uses the first
> `20` of these (in selection order) as the final discussion-statement set
> (`§Discussion Statements`). Keeping the extra 10 in reserve allowed a later
> substitution without re-running the whole selection procedure.


In [ ]:
df_order = pl.DataFrame({"idx": selected_ids, "selection_order": range(len(selected_ids))})

df_selected = (
    df_preds
    .filter(pl.col("idx").is_in(selected_ids))
    .join(df_order, on="idx", how="inner")
    .sort("selection_order")
)
df_selected.select(["idx", "statement"] + FEATURE_COLS)


### Diagnostics

Sanity-check the *spread* of the selected statements in feature space:

1. Higher `min_pairwise_dist` -> better spread (this is exactly what maximin optimizes).
2. Larger per-feature variance/range -> better coverage of each individual feature.
3. PCA explained-variance ratios closer to uniform -> more isotropic (not
   concentrated along one axis) spread.


In [ ]:
from scipy.spatial.distance import pdist
from sklearn.decomposition import PCA


def diversity_report(df: pl.DataFrame, idx: list[int], feature_cols: list[str]) -> dict:
    """Compute simple diversity diagnostics on a selected subset of rows."""
    X = df.filter(pl.col("idx").is_in(idx)).select(feature_cols).to_numpy().astype(float)
    dists = pdist(X, metric="euclidean")
    pca = PCA().fit(X)
    return {
        "min_pairwise_dist": float(dists.min()),
        "mean_pairwise_dist": float(dists.mean()),
        "median_pairwise_dist": float(np.median(dists)),
        "feature_variance": dict(zip(feature_cols, X.var(axis=0).tolist())),
        "feature_range": dict(zip(feature_cols, (X.max(axis=0) - X.min(axis=0)).tolist())),
        "pca_explained_ratio": pca.explained_variance_ratio_.tolist(),
    }


diversity_report(df_preds, selected_ids, FEATURE_COLS)


### Writing the final statement configs

The cell below writes one YAML config per selected statement to
`configs/statement/`, in the format consumed by the simulator. **This is the
step that originally produced the committed experiment configs** — re-running
it with a different `selected_ids` would overwrite them, so it is guarded by
an explicit `raise` that must be removed intentionally before re-running.


In [ ]:
raise RuntimeError(
    "Guard against accidental 'Run All': this cell overwrites the committed "
    "statement configs. Remove this line only if you intend to regenerate them."
)


In [ ]:
from textwrap import dedent

import yaml

STAT_FIELDS = [
    "label_ground_truth",
    "doc_predicted_label",
    "doc_prob",
    "doc_accuracy",
    "other_frac_predicted_label",
    "other_accuracy",
    "other_median_prob",
    "consensus_score",
]


# Force a literal block scalar for multiline strings (readability in the output YAML).
def _str_representer(dumper, data):
    if "\n" in data:
        return dumper.represent_scalar("tag:yaml.org,2002:str", data, style="|")
    return dumper.represent_scalar("tag:yaml.org,2002:str", data)


yaml.add_representer(str, _str_representer)


def write_statement_configs(
    df: pl.DataFrame,
    selected_ids: list[int],
    out_dir: Path = Path("configs/statement"),
) -> list[Path]:
    """Write one YAML config per selected statement, numbered per label (true_0, false_0, ...)."""
    out_dir.mkdir(parents=True, exist_ok=True)

    description = dedent("""\
    Auto-generated from the maximin selection pipeline.

    Explaining fields in `stats` (the "doctor" LLM is m42-health/Llama3-Med42-8B):
    - label_ground_truth: statement label (binary, 1=True, 0=False)
    - doc_predicted_label: predicted label by the doctor LLM (binary, 1=True, 0=False)
    - doc_prob: probability assigned by the doctor LLM to the statement being true
    - doc_accuracy: accuracy of the doctor LLM's prediction vs. ground truth
    - other_frac_predicted_label: fraction of other LLMs predicting the statement as true
    - other_accuracy: accuracy of other LLMs' predictions vs. ground truth
    - other_median_prob: median probability from other LLMs that the statement is true
    - consensus_score: agreement between the doctor LLM and other LLMs' probabilities (not accuracy)

    Statements were selected via maximin over: label_ground_truth, doc_predicted_label,
    doc_accuracy, other_accuracy, consensus_score.
    """).strip()

    df_filtered = df.filter(pl.col("idx").is_in(selected_ids)).sort("selection_order")

    label_counters = {"false": 0, "true": 0}
    paths: list[Path] = []
    for row in df_filtered.to_dicts():
        label_key = "true" if row["label_ground_truth"] else "false"
        cfg_id = f"{label_key}_{label_counters[label_key]}"
        label_counters[label_key] += 1

        cfg = {
            "defaults": ["base", "_self_"],
            "id": cfg_id,
            "statement": row["statement"],
            "label": {"correct": bool(row["label_ground_truth"]), "neither": False, "negated": False},
            "description": description,
            "stats": {
                k: (round(float(row[k]), 5) if isinstance(row[k], (int, float)) else row[k])
                for k in STAT_FIELDS
            },
        }
        path = out_dir / f"{cfg_id}.yaml"
        with path.open("w", encoding="utf-8") as f:
            yaml.dump(cfg, f, sort_keys=False, allow_unicode=True, default_flow_style=False)
        paths.append(path)

    return paths


write_statement_configs(df=df_selected, selected_ids=selected_ids)
